In [0]:
from pyspark.sql.functions import when, col, lit, sum as spark_sum, count

## Rates per TPMA

In [0]:
rates_data = spark.read.parquet("/Volumes/nhp/inputs_data/files/dev/lad23cd/rates.parquet")
display(rates_data)

In [0]:
national_rates_data = (
    rates_data
    .filter((col("lad23cd") == "national") & col("fyear").between(201516, 202324))
    .withColumn("numerator", col("crude_rate") * col("denominator"))
    .drop("lad23cd")
)

display(national_rates_data)

## Totals per activity type and by mitigation

### apc

### opa

In [0]:
opa = spark.table("nhp.raw_data.opa")
opa_mitig = spark.table("nhp.raw_data.opa_mitigators")

In [0]:
opa_join_cols = ["attendkey", "fyear", "provider"]

opa = opa.filter(col("fyear").between(201516, 202324))

opa_mitig = opa.join(
    opa_mitig.select(*opa_join_cols).distinct(),
    on=opa_join_cols,
    how="left_semi"
).withColumn("mitigable", lit("mitigable"))

opa_usual = opa.join(
    opa_mitig.select(*opa_join_cols).distinct(),
    on=opa_join_cols,
    how="left_anti"
).withColumn("mitigable", lit("usual"))

opa_by_mitig = opa_mitig.unionByName(opa_usual)

opa_total = opa.withColumn("mitigable", lit("all"))

opa_final = opa_by_mitig.unionByName(opa_total)

opa_counts = (
    opa_final
    .groupBy("fyear", "mitigable")
    .agg(count("attendkey")))


display(opa_counts)

## ED

In [0]:
ed_data = spark.table("nhp.raw_data.ecds")

In [0]:
ed_data = ed_data.filter(col("fyear").between(201516, 202324))

ed_mitig = (
    ed_data
    .withColumn(
        "mitigable",
        when(
            (col("is_frequent_attender")) |
            (col("is_left_before_treatment")) |
            (col("is_low_cost_referred_or_discharged")) |
            (col("is_discharged_no_treatment")),
            "mitigable"
        ).otherwise("usual")
    )
)

ed_all = ed_data.withColumn("mitigable", lit("all"))

ed_final = ed_mitig.unionByName(ed_all)

ed_counts = ed_final.groupBy("fyear", "mitigable").count()

display(ed_counts)